In [2]:
!pip install pymongo
import pymongo
from pymongo import MongoClient
from pymongo.errors import ServerSelectionTimeoutError, PyMongoError
from google.colab import userdata

# Get MongoDB link from Colab secret
mongo_link = userdata.get("MONGO_CONNECTION_STRING")

try:
    client = MongoClient(mongo_link)

    # The ping command is cheap and does not require auth.
    client.admin.command('ping')
    print("MongoDB connection successful!")

except ServerSelectionTimeoutError as err:
    print(f"MongoDB connection failed: Server Selection Timeout. This often indicates network issues or incorrect IP whitelisting in MongoDB Atlas. Error: {err}")
except PyMongoError as err:
    print(f"An unexpected PyMongo error occurred during connection: {err}")
except Exception as err:
    print(f"An unexpected error occurred: {err}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 22.8 MB/s eta 0:00:00
MongoDB connection successful!


In [3]:
{
    "delivery_id": "DL1025",
    "order_id": "ORD2041",
    "delivery_status": "Delayed",
    "route_distance_km": 18.5,
    "manual_route_override_count": 3,
    "customer_feedback": {
        "rating": 2,
        "comment": "Delivery arrived late"
    },
    "incident_history": [
        {
            "incident_type": "Traffic Delay",
            "severity": "Medium"
        }
    ]
}

{'delivery_id': 'DL1025',
 'order_id': 'ORD2041',
 'delivery_status': 'Delayed',
 'route_distance_km': 18.5,
 'manual_route_override_count': 3,
 'customer_feedback': {'rating': 2, 'comment': 'Delivery arrived late'},
 'incident_history': [{'incident_type': 'Traffic Delay',
   'severity': 'Medium'}]}

In [5]:
delivery_document = {
    "delivery_id": "DL2001",
    "delivery_status": "Completed",
    "route_distance_km": 14.2,
    "manual_route_override_count": 1
}

db = client.mydatabase # Replace 'mydatabase' with your actual database name
db.deliveries.insert_one(delivery_document)

InsertOneResult(ObjectId('6a074dec214d3dad9fbf5faf'), acknowledged=True)

In [6]:
results = db.deliveries.find({
    "delivery_status": "Delayed"
})

for record in results:
    print(record)

In [7]:
db.deliveries.update_one(
    {"delivery_id": "DL2001"},
    {
        "$set": {
            "delivery_status": "Completed"
        }
    }
)

UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff000000000000010e'), 'opTime': {'ts': Timestamp(1778863653, 1), 't': 270}, 'nModified': 0, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1778863653, 1), 'signature': {'hash': b'\xa9\xfdDi\xc4\xbb#]\x7f\xc6\x9a:\xd9\x95\xbd%\xa1\xba\x8d\x18', 'keyId': 7581452816082796548}}, 'operationTime': Timestamp(1778863653, 1), 'updatedExisting': True}, acknowledged=True)

In [8]:
db.deliveries.delete_one({
    "delivery_id": "DL2001"
})

DeleteResult({'n': 1, 'electionId': ObjectId('7fffffff000000000000010e'), 'opTime': {'ts': Timestamp(1778863667, 1), 't': 270}, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1778863667, 1), 'signature': {'hash': b"\xbf\x94\x08\xaf\x19\xc4f\x14e\x82[\xa1\xb8\x9f1V9\x96'\xa3", 'keyId': 7581452816082796548}}, 'operationTime': Timestamp(1778863667, 1)}, acknowledged=True)

In [9]:
pipeline = [
    {
        "$match": {
            "delivery_status": "Failed"
        }
    },
    {
        "$group": {
            "_id": "$hub_id",
            "total_failed_deliveries": {
                "$sum": 1
            },
            "avg_route_distance": {
                "$avg": "$route_distance_km"
            }
        }
    }
]

results = db.deliveries.aggregate(pipeline)

for result in results:
    print(result)

In [11]:
pipeline = [
    {
        "$group": {
            "_id": "$severity",
            "total_complaints": {
                "$sum": 1
            },
            "avg_compensation": {
                "$avg": "$compensation_amount"
            }
        }
    }
]

results = db.complaints.aggregate(pipeline)

for result in results:
    print(result)

In [12]:
db.deliveries.create_index("delivery_status")
db.deliveries.create_index("driver_id")
db.deliveries.create_index("vehicle_id")
db.complaints.create_index("severity")

'severity_1'

In [13]:
db.deliveries.create_index([
    ("delivery_status", 1),
    ("hub_id", 1)
])

'delivery_status_1_hub_id_1'

In [14]:
explain_result = db.deliveries.find(
    {"delivery_status": "Failed"}
).explain()

print(explain_result)

{'explainVersion': '1', 'queryPlanner': {'namespace': 'mydatabase.deliveries', 'parsedQuery': {'delivery_status': {'$eq': 'Failed'}}, 'indexFilterSet': False, 'queryHash': 'CC376D25', 'planCacheShapeHash': 'CC376D25', 'planCacheKey': '227DD467', 'optimizationTimeMillis': 0, 'maxIndexedOrSolutionsReached': False, 'maxIndexedAndSolutionsReached': False, 'maxScansToExplodeReached': False, 'prunedSimilarIndexes': False, 'winningPlan': {'isCached': False, 'stage': 'FETCH', 'inputStage': {'stage': 'IXSCAN', 'keyPattern': {'delivery_status': 1}, 'indexName': 'delivery_status_1', 'isMultiKey': False, 'multiKeyPaths': {'delivery_status': []}, 'isUnique': False, 'isSparse': False, 'isPartial': False, 'indexVersion': 2, 'direction': 'forward', 'indexBounds': {'delivery_status': ['["Failed", "Failed"]']}}}, 'rejectedPlans': [{'isCached': False, 'stage': 'FETCH', 'inputStage': {'stage': 'IXSCAN', 'keyPattern': {'delivery_status': 1, 'hub_id': 1}, 'indexName': 'delivery_status_1_hub_id_1', 'isMultiK

In [16]:
pipeline = [
    {
        "$match": {
            "delivery_status": "Failed"
        }
    },
    {
        "$group": {
            "_id": "$hub_id",
            "total_failed_deliveries": {"$sum": 1},
            "avg_route_distance": {"$avg": "$route_distance_km"}
        }
    },
    {
        "$sort": {
            "total_failed_deliveries": -1
        }
    }
]

results = db.deliveries.aggregate(pipeline)

for result in results:
    print(result)